# 🏦 Loan Approval Prediction System
**AIML Summer Internship 2026 — IIHMF, MNNIT Allahabad**

---

## 📦 Phase 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
import pickle

print('✅ All libraries imported successfully!')

## 📂 Phase 2: Load Dataset

In [ ]:
# Upload train.csv when prompted
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('train.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Basic Statistics ===')
df.describe()

## 🧹 Phase 3: Data Preprocessing

In [ ]:
# Missing values check
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Fill missing values
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)

print('✅ Missing values handled!')
print(df.isnull().sum())

In [ ]:
# Remove duplicates
print('Duplicates before:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('Duplicates after:', df.duplicated().sum())
print('✅ Duplicates removed!')

In [ ]:
# Drop Loan_ID (not useful)
df.drop('Loan_ID', axis=1, inplace=True)
print('✅ Loan_ID dropped!')
print('Current columns:', list(df.columns))

In [ ]:
# Encode categorical columns
le = LabelEncoder()
cat_cols = ['Gender', 'Married', 'Dependents', 'Education',
            'Self_Employed', 'Property_Area', 'Loan_Status']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('✅ Categorical encoding done!')
df.head()

## 📊 Phase 4: Exploratory Data Analysis (EDA)

In [ ]:
# 1. Target Distribution
plt.figure(figsize=(6,4))
df['Loan_Status'].value_counts().plot(kind='bar', color=['#e74c3c','#2ecc71'], edgecolor='black')
plt.title('Loan Approval Distribution\n(0 = Not Approved, 1 = Approved)', fontsize=13)
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Correlation Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Univariate Analysis - Histograms for numerical columns
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
df[num_cols].hist(bins=20, figsize=(12,8), color='steelblue', edgecolor='black')
plt.suptitle('Univariate Analysis - Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3b. Univariate Analysis - KDE (Smooth Distribution Curves)
fig, axes = plt.subplots(2, 2, figsize=(12,8))
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, bins=20, color='steelblue', edgecolor='black', ax=ax)
    ax.set_title(f'Distribution of {col} (with KDE)')

plt.suptitle('Univariate Analysis - KDE Plots', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Boxplots - Outlier Detection
fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.boxplot(x='Loan_Status', y='ApplicantIncome', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Applicant Income vs Loan Status')
sns.boxplot(x='Loan_Status', y='LoanAmount', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Loan Amount vs Loan Status')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Credit History Impact
plt.figure(figsize=(6,4))
sns.countplot(x='Credit_History', hue='Loan_Status', data=df, palette='Set1')
plt.title('Credit History vs Loan Status')
plt.xlabel('Credit History (0=Bad, 1=Good)')
plt.ylabel('Count')
plt.legend(title='Loan Status', labels=['Not Approved', 'Approved'])
plt.tight_layout()
plt.show()

In [ ]:
# 6. Property Area vs Loan Status
plt.figure(figsize=(6,4))
sns.countplot(x='Property_Area', hue='Loan_Status', data=df, palette='Set2')
plt.title('Property Area vs Loan Status')
plt.xlabel('Property Area (0=Rural, 1=Semiurban, 2=Urban)')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Scatter Plot - Income vs Loan Amount
plt.figure(figsize=(8,5))
sns.scatterplot(x='ApplicantIncome', y='LoanAmount', hue='Loan_Status',
                data=df, palette='Set1', alpha=0.7)
plt.title('Applicant Income vs Loan Amount')
plt.tight_layout()
plt.show()

## ⚙️ Phase 5: Feature Engineering

In [ ]:
# New Feature: Total Income
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# Log Transformation to reduce skewness
df['LoanAmount_log'] = np.log(df['LoanAmount'] + 1)
df['Total_Income_log'] = np.log(df['Total_Income'] + 1)

# EMI feature: Loan Amount per month
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']

# Balance Income after EMI
df['Balance_Income'] = df['Total_Income'] - (df['EMI'] * 1000)

# Drop original skewed columns
df.drop(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Total_Income'], axis=1, inplace=True)

print('✅ Feature Engineering done!')
print('New shape:', df.shape)
df.head()

## 🤖 Phase 6: Model Building

In [ ]:
# Train-Test Split
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('\nFeatures used:', list(X.columns))

In [ ]:
# Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
print('✅ Logistic Regression trained!')

In [ ]:
# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('✅ Random Forest trained!')

In [ ]:
# Model 3: XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
print('✅ XGBoost trained!')

## 📈 Phase 7: Model Evaluation

In [ ]:
# Compare all 3 models
models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results)
print('=== Model Comparison ===')
results_df

In [ ]:
# Detailed Classification Report
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Not Approved', 'Approved']))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16,4))

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Not Approved', 'Approved'],
                yticklabels=['Not Approved', 'Approved'])
    axes[idx].set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.4f}')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices - All Models', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8,6))

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0,1],[0,1],'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - All Models')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feat_importance = pd.Series(rf.feature_importances_, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False)

plt.figure(figsize=(8,5))
feat_importance.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Model Comparison Bar Chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12,6))

for i, (_, row) in enumerate(results_df.iterrows()):
    ax.bar(x + i*width, row[metrics], width, label=row['Model'])

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Comparison - All Metrics')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 💾 Phase 8: Save Best Model

In [ ]:
# Save best model (Random Forest)
best_model = rf  # Change if XGBoost performs better

with open('loan_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save feature names for Flask app
feature_names = list(X.columns)
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print('✅ Model saved as loan_model.pkl')
print('✅ Feature names saved as feature_names.pkl')
print('Features:', feature_names)

In [ ]:
# Download model files
from google.colab import files
files.download('loan_model.pkl')
files.download('feature_names.pkl')
print('✅ Files downloaded!')

## ✅ Summary

In [ ]:
print('=' * 50)
print('   LOAN APPROVAL PREDICTION - FINAL SUMMARY')
print('=' * 50)
print(f'Total Samples: {len(df)}')
print(f'Features Used: {X.shape[1]}')
print(f'Train Size: {X_train.shape[0]}')
print(f'Test Size: {X_test.shape[0]}')
print()
print(results_df.to_string(index=False))
print()
best_row = results_df.loc[results_df['Accuracy'].idxmax()]
print(f'🏆 Best Model: {best_row["Model"]} (Accuracy: {best_row["Accuracy"]})')